<a href="https://colab.research.google.com/github/aronwilbert/COMP8420-Group-L-Healthcare/blob/main/COMP8420_Group_L_Healthcare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === CELL 1: FORCE SYNCHRONIZE PYTORCH AND LLM ARCHITECTURES ===
!pip install --upgrade --force-reinstall numpy pandas scipy scikit-learn textblob spacy datasets gliner transformers torch torchvision
!python -m spacy download en_core_web_md
!python -m textblob.download_corpora lite
!pip install contractions

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.0/625.0 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

^C
^C
^C


In [ ]:
# === CELL 2: MODERNISED NLP EXECUTION CELL ===
import pandas as pd
from textblob import TextBlob
from sklearn.metrics.pairwise import cosine_similarity
import spacy
from gliner import GLiNER
from datasets import load_dataset
import contractions

In [ ]:
# 1. Load the raw master dataset
master_dataset = load_dataset("xDAN-datasets/ChatDoctor_HealthCareMagic_112k", split="train")
print(f"📊 Initial Raw Master Dataset Samples: {len(master_dataset)}")

# --- VISUAL 1: DISPLAY RAW DATASET BEFORE STRUCTURAL FILTERING ---
print("\n--- 🛑 RAW DATASET PREVIEW (BEFORE FILTERING / RAW INGESTION) ---")
df_raw_preview = pd.DataFrame(master_dataset.select(range(5)))
display(df_raw_preview)

# 2. Filter out rows where 'input' or 'output' are missing or completely empty
clean_master = master_dataset.filter(
    lambda x: x["input"] is not None and x["output"] is not None and
              str(x["input"]).strip() != "" and str(x["output"]).strip() != ""
)
print(f"\n🧹 Structurally Valid Master Dataset Samples (After NA/Blank Removal): {len(clean_master)}")
print(f"❌ Total Defective Rows Filtered: {len(master_dataset) - len(clean_master)}")

# 3. Securely isolate EXACTLY 5,000 clean samples for the project
dataset = clean_master.select(range(5000))
print(f"✅ Final Target Samples Isolated for Analysis Pipeline: {len(dataset)}")

# --- VISUAL 2: DISPLAY SUBSET BEFORE TEXT PREPROCESSING ---
df_clean = pd.DataFrame({
    'input': [row['input'] for row in dataset],
    'output': [row['output'] for row in dataset]
})

print("\n--- 📋 VERIFIED BASELINE DATASET SNAPSHOT (PRE-TEXT CLEANING) ---")
display(df_clean.head(5))

In [ ]:
import pandas as pd
import re

print("================================================================================")
print("🧼 EXECUTING FULL-SCALE ENGINE: ANOMALIES ELIMINATED (5,000 ROWS)")
print("================================================================================")

def execute_final_period_logic_v5(text):
    if not isinstance(text, str):
        return ""

    # Standardize to lowercase immediately
    text = text.lower()

    # 🔴 STEP 1: HARD LOCK KEY EXCEPTIONS FIRST
    text = re.sub(r'i\.e\.', '_LOCK_IE_', text)
    text = re.sub(r'\bi\.e\b', '_LOCK_IE_', text)
    text = re.sub(r'e\.g\.', '_LOCK_EG_', text)
    text = re.sub(r'\be\.g\b', '_LOCK_EG_', text)
    text = re.sub(r'dr\.', '_LOCK_DR_', text)

    # STEP 2: Strip brackets completely
    text = re.sub(r'[\(\)]', ' ', text)

    # 🔥 FIXED STEP 2B: Un-fuse text/numbers stuck together by removed brackets (e.g., "1 cbc" vs "1)cbc")
    text = re.sub(r'([a-z])(\d)', r'\1 \2', text)
    text = re.sub(r'(\d)([a-z])', r'\1 \2', text)

    # STEP 2C: Strip ALL question marks universally
    text = text.replace('?', ' ')

    # STEP 3: Mask and protect vital clinical decimals and dates
    text = re.sub(r'(\d+)\.(\d+)', r'\1_SAFE_DEC_\2', text)

    # STEP 4: Collapse multiple repeating dots (ellipses) to a space
    text = re.sub(r'\.{2,}', ' ', text)

    # STEP 5: Split fused sentence text/number boundaries (e.g., "medicine.it")
    text = re.sub(r'([a-z0-9])\.([a-z])', r'\1 \2', text)

    # STEP 6: Clean up any leftover floating/trailing punctuation noise (, ; ! or terminal dots)
    text = re.sub(r'[.,;!]+(?=\s|$)', '', text)

    # 🔥 FIXED STEP 6B: Clear out stray standalone periods left by floating dots
    text = re.sub(r'\s\.\s', ' ', text)

    # 🟢 STEP 7: RESTORE ORIGINAL PROTECTED EXCEPTIONS
    text = text.replace('_LOCK_IE_', 'i.e')
    text = text.replace('_LOCK_EG_', 'e.g')
    text = text.replace('_LOCK_DR_', 'dr.')
    text = text.replace('_SAFE_DEC_', '.')

    # STEP 8: Eliminate structural double spacing noise
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the absolute cleanest pipeline variant to all 5,000 rows
df_clean['clean_input_periods'] = df_clean['input'].astype(str).apply(execute_final_period_logic_v5)

print("🎉 SUCCESS: Anomalies completely squashed across all 5,000 rows!")
print("=" * 80)
print("📊 VERIFICATION PANEL (AUDITING PREVIOUS PROBLEM ROWS)")
print("=" * 80)

# Target specifically the anomaly rows to verify the fixes immediately
for idx in [6, 38]:
    print(f"\n📍 DATASET ROW INDEX: {idx}")
    print("-" * 70)
    print(f"🔴 ORIGINAL INPUT :\n   \"{df_clean['input'].iloc[idx][:200]}...\"")
    print(f"🟢 BULLETPROOF CLEAN :\n   \"{df_clean['clean_input_periods'].iloc[idx][:200]}...\"")
    print("=" * 80)

🧼 EXECUTING FULL-SCALE ENGINE: ANOMALIES ELIMINATED (5,000 ROWS)
🎉 SUCCESS: Anomalies completely squashed across all 5,000 rows!
📊 VERIFICATION PANEL (AUDITING PREVIOUS PROBLEM ROWS)

📍 DATASET ROW INDEX: 6
----------------------------------------------------------------------
🔴 ORIGINAL INPUT :
   "sir, MY uncle has ILD-Interstitial Lung disease.from my research over google i found that the cause is important to be known. the cause wat i feel is due to breakage of acid bottle in bathroom which c..."
🟢 BULLETPROOF CLEAN :
   "sir my uncle has ild-interstitial lung disease from my research over google i found that the cause is important to be known the cause wat i feel is due to breakage of acid bottle in bathroom which cau..."

📍 DATASET ROW INDEX: 38
----------------------------------------------------------------------
🔴 ORIGINAL INPUT :
   "Dear doctor, i am 33 male recently i have undergone pre employment medical check up and it was found that i have sgpt- 194 except this all test 

==== NEXT TASK IS TO FIX WORDS =====

In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForMaskedLM
import pandas as pd
import re

print("================================================================================")
print("🤖 LOADING DEEP LEARNING CLINICAL-BERT COGNITIVE LAYER")
print("================================================================================")

# Step 1: Initialize the pipeline with pre-trained ClinicalBERT weights from Hugging Face
model_name = "medicalai/ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

# Use GPU acceleration if your Google Colab instance has it turned on
device = 0 if torch.cuda.is_available() else -1
nlp_fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer, device=device)

def clinical_bert_spell_fixer(text, target_typo, expected_mask_token="[MASK]"):
    """
    Leverages ClinicalBERT's pre-trained vocabulary to contextually swap out
    a known typo with its highest-probability clinical alternative.
    """
    if target_typo not in text:
        return text

    # Swap out the target typo with BERT's mask token
    masked_text = re.sub(rf'\b{target_typo}\b', expected_mask_token, text)

    try:
        # Ask ClinicalBERT to fill the blank based on clinical context
        predictions = nlp_fill_mask(masked_text)

        # Pull out the highest probability token predicted by the model
        if isinstance(predictions, list) and len(predictions) > 0:
            # If multiple masks exist, pick the first prediction array
            top_prediction = predictions[0] if isinstance(predictions[0], dict) else predictions[0][0]
            corrected_token = top_prediction['token_str'].strip()

            # Reconstruct the text with ClinicalBERT's replacement
            return text.replace(target_typo, corrected_token)
    except Exception as e:
        # Fallback to original text if context lengths or tokens mismatch
        return text
    return text

print("🎉 CLINICAL-BERT MODEL ONLINE! EXECUTING PREVIEW CHECK...")
print("=" * 80)

# Step 2: Grab problem rows directly from your dataset to audit live
sample_row_3 = "lump under left nipple and stomach pain male hi,i have recently noticed a few weeks ago a lump under my nipple it hurts to touch and is about the size of a quarter also i have bern experiencing stomach pains that prevent me from eating i immediatly feel full and have extreme pain please help"

print(f"🔴 RAW DATASET SAMPLE:\n   \"{sample_row_3}\"")
print("-" * 80)

# Run ClinicalBERT against the typo 'immediatly'
clean_step_1 = clinical_bert_spell_fixer(sample_row_3, "immediatly")
# Run ClinicalBERT against the contextual typing slip 'bern'
final_clean = clinical_bert_spell_fixer(clean_step_1, "bern")

print(f"🟢 CLINICAL-BERT CORRECTED OUTPUT:\n   \"{final_clean}\"")
print("=" * 80)

🤖 LOADING DEEP LEARNING CLINICAL-BERT COGNITIVE LAYER


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/542M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie distilbert.embeddings.word_embeddings.weight to vocab_projector.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


🎉 CLINICAL-BERT MODEL ONLINE! EXECUTING PREVIEW CHECK...
🔴 RAW DATASET SAMPLE:
   "lump under left nipple and stomach pain male hi,i have recently noticed a few weeks ago a lump under my nipple it hurts to touch and is about the size of a quarter also i have bern experiencing stomach pains that prevent me from eating i immediatly feel full and have extreme pain please help"
--------------------------------------------------------------------------------
🟢 CLINICAL-BERT CORRECTED OUTPUT:
   "lump under left nipple and stomach pain male hi,i have recently noticed a few weeks ago a lump under my nipple it hurts to touch and is about the size of a quarter also i have been experiencing stomach pains that prevent me from eating i do feel full and have extreme pain please help"


In [ ]:
# Ensure master data list is isolated
raw_documents = df_clean_final['input'].astype(str).tolist()

# =====================================================================
# 📋 STEP 1: PRE-CLEANING BASELINE ANALYSIS (STOPWORDS REMOVED FOR MATH ONLY)
# =====================================================================
print("=== 📋 STEP 1: PRE-CLEANING BASELINE PROFILE ===")

pre_tfidf = TfidfVectorizer(stop_words='english', max_features=10)
pre_tfidf_matrix = pre_tfidf.fit_transform(raw_documents)
df_pre_tfidf = pd.DataFrame({'Pre-Clean Word': pre_tfidf.get_feature_names_out(), 'TF-IDF Score': pre_tfidf_matrix.mean(axis=0).A1}).sort_values(by='TF-IDF Score', ascending=False).reset_index(drop=True)

pre_count = CountVectorizer(stop_words='english', ngram_range=(2, 3), max_features=10)
pre_count_matrix = pre_count.fit_transform(raw_documents)
df_pre_ngrams = pd.DataFrame({'Pre-Clean Phrase': pre_count.get_feature_names_out(), 'Frequency Count': pre_count_matrix.sum(axis=0).A1}).sort_values(by='Frequency Count', ascending=False).reset_index(drop=True)

html_pre = """
<div style="display: flex;">
    <div style="margin-right: 50px;">
        <h3>📊 Word Importance Baseline</h3>
        {}
    </div>
    <div>
        <h3>🏷️ Multi-Word Phrase Baseline</h3>
        {}
    </div>
</div>
""".format(df_pre_tfidf.to_html(index=False), df_pre_ngrams.to_html(index=False))
display_html(html_pre, raw=True)


# =====================================================================
# 🧼 STEP 2: CAREFULLY ORDERED DATA CLEANING PASS
# =====================================================================
print("\n" + "="*80 + "\n=== 🧼 STEP 2: EXECUTING CAREFULLY ORDERED PIPELINE ===")

def ultimate_medical_cleaner(text):
    if not isinstance(text, str): return ""

    # 1. Expand contractions to safeguard downstream word structures
    contractions = {
        r"can't": "cannot", r"don't": "do not", r"it's": "it is", r"i'm": "i am",
        r"doesn't": "does not", r"didn't": "did not", r"he's": "he is", r"she's": "she is"
    }
    for contraction, expansion in contractions.items():
        text = re.sub(contraction, expansion, text, flags=re.IGNORECASE)

    # 2. Add spaces after jammed conversational punctuation
    text = re.sub(r'([,;\.\!\?])([A-Za-z])', r'\1 \2', text)

    # 3. Strip greetings and sign-offs at boundaries
    text = re.sub(r'^\b(hi+|hello+|hey|dear)\b\s*\b(doc|doctor)?\b[,\s\.]*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(thank\s*you|thanks)\s*\b(doc|doctor)?\b[!\.]*$', '', text, flags=re.IGNORECASE)

    # 4. Extract and mask complex age features
    text = re.sub(r'\b\d+\s*(years?|yrs?|months?|mos?|mths?)\s*(old)?\b', '[REDACTED_AGE]', text, flags=re.IGNORECASE)

    # 5. Targeted Punctuation Swap: Replace punctuation with spaces, KEEPING hyphens (-) and brackets ([])
    text = re.sub(r'[^\w\s\-\[\]]', ' ', text)

    # 6. Compress formatting layouts into clear singular spaces
    return re.sub(r'\s+', ' ', text).strip()

# Apply to your dataset
df_clean_final['clean_input'] = df_clean_final['input'].apply(ultimate_medical_cleaner)
clean_documents = df_clean_final['clean_input'].tolist()

print("✅ Master Dataset Cleaned Successfully.")


# =====================================================================
# ✅ STEP 3: POST-CLEANING VERIFICATION ANALYSIS
# =====================================================================
print("\n" + "="*80 + "\n=== ✅ STEP 3: POST-CLEANING VERIFICATION PROFILE ===")

post_tfidf = TfidfVectorizer(stop_words='english', max_features=10, token_pattern=r'(?u)\b\w+\b|\[\w+\]')
post_tfidf_matrix = post_tfidf.fit_transform(clean_documents)
df_post_tfidf = pd.DataFrame({'Cleaned Term': post_tfidf.get_feature_names_out(), 'TF-IDF Score': post_tfidf_matrix.mean(axis=0).A1}).sort_values(by='TF-IDF Score', ascending=False).reset_index(drop=True)

post_count = CountVectorizer(stop_words='english', ngram_range=(2, 3), max_features=10, token_pattern=r'(?u)\b\w+\b|\[\w+\]')
post_count_matrix = post_count.fit_transform(clean_documents)
df_post_ngrams = pd.DataFrame({'Cleaned Phrase': post_count.get_feature_names_out(), 'Frequency Count': post_count_matrix.sum(axis=0).A1}).sort_values(by='Frequency Count', ascending=False).reset_index(drop=True)

html_post = """
<div style="display: flex;">
    <div style="margin-right: 50px;">
        <h3>📊 Cleaned Word Importance</h3>
        {}
    </div>
    <div>
        <h3>🏷️ Cleaned Multi-Word Phrases</h3>
        {}
    </div>
</div>
""".format(df_post_tfidf.to_html(index=False), df_post_ngrams.to_html(index=False))
display_html(html_post, raw=True)

display(df_clean_final[['input', 'clean_input']].head(3))

=== 📋 STEP 1: PRE-CLEANING BASELINE PROFILE ===


Pre-Clean Word,TF-IDF Score
pain,0.188483
hi,0.136012
old,0.131301
like,0.129531
doctor,0.123912
days,0.119800
years,0.119387
just,0.116030
ago,0.098343
right,0.097079



=== 🧼 STEP 2: EXECUTING CAREFULLY ORDERED PIPELINE ===
✅ Master Dataset Cleaned Successfully.

=== ✅ STEP 3: POST-CLEANING VERIFICATION PROFILE ===


Cleaned Term,TF-IDF Score
[redacted_age],0.230689
pain,0.179547
like,0.127874
days,0.119208
just,0.114804
doctor,0.106497
ago,0.096186
right,0.095093
time,0.095007
left,0.081199


,input,clean_input
0,I woke up this morning feeling the whole room ...,I woke up this morning feeling the whole room ...
1,My baby has been pooing 5-6 times a day for a ...,My baby has been pooing 5-6 times a day for a ...
2,"Hello, My husband is taking Oxycodone due to a...",My husband is taking Oxycodone due to a broken...


In [ ]:
import re

# Pick an excellent sample row that contains contractions, mashed punctuation, greetings, and ages
sample_raw_text = df_clean_final['input'].iloc[3]

print("================================================================================")
print("🔬 STEP-BY-STEP DATA ENGINEERING TRACE PIPELINE")
print("================================================================================\n")

print(f"🔴 [0. ORIGINAL RAW TEXT]:\n{sample_raw_text}\n")
print("-" * 80)

# --- STEP 1: CONTRACTION EXPANSION ---
text_step1 = sample_raw_text
contractions = {
    r"can't": "cannot", r"don't": "do not", r"it's": "it is", r"i'm": "i am",
    r"doesn't": "does not", r"didn't": "did not", r"he's": "he is", r"she's": "she is"
}
for contraction, expansion in contractions.items():
    text_step1 = re.sub(contraction, expansion, text_step1, flags=re.IGNORECASE)

print(f"🔄 [1. AFTER CONTRACTION EXPANSION] (e.g., don't -> do not):\n{text_step1}\n")
print("-" * 80)

# --- STEP 2: PUNCTUATION SPACING ---
text_step2 = re.sub(r'([,;\.\!\?])([A-Za-z])', r'\1 \2', text_step1)

print(f"🔄 [2. AFTER PUNCTUATION SPACING ADJUSTMENT] (e.g., Hi,I -> Hi, I):\n{text_step2}\n")
print("-" * 80)

# --- STEP 3: GREETING & SIGN-OFF REMOVAL ---
text_step3 = re.sub(r'^\b(hi+|hello+|hey|dear)\b\s*\b(doc|doctor)?\b[,\s\.]*', '', text_step2, flags=re.IGNORECASE)
text_step3 = re.sub(r'\b(thank\s*you|thanks)\s*\b(doc|doctor)?\b[!\.]*$', '', text_step3, flags=re.IGNORECASE)

print(f"🔄 [3. AFTER RULE-BASED GREETING & SIGN-OFF REMOVAL]:\n{text_step3}\n")
print("-" * 80)

# --- STEP 4: AGE ANONYMIZATION ---
text_step4 = re.sub(r'\b\d+\s*(years?|yrs?|months?|mos?|mths?)\s*(old)?\b', '[REDACTED_AGE]', text_step3, flags=re.IGNORECASE)

print(f"🔄 [4. AFTER PATTERN-BASED AGE REDACTION]:\n{text_step4}\n")
print("-" * 80)

# --- STEP 5: TARGETED PUNCTUATION FILTER ---
text_step5 = re.sub(r'[^\w\s\-\[\]]', ' ', text_step4)

print(f"🔄 [5. AFTER PUNCTUATION NORMALIZATION] (Keeps hyphens and brackets):\n{text_step5}\n")
print("-" * 80)

# --- STEP 6: WHITESPACE COMPRESSION ---
text_step6 = re.sub(r'\s+', ' ', text_step5).strip()

print(f"🟢 [6. FINAL CLEAN OUTPUT SENTENCE FOR CHATBOT]:\n{text_step6}\n")
print("================================================================================")

🔬 STEP-BY-STEP DATA ENGINEERING TRACE PIPELINE

🔴 [0. ORIGINAL RAW TEXT]:
lump under left nipple and stomach pain (male) Hi,I have recently noticed a few weeks ago a lump under my nipple, it hurts to touch and is about the size of a quarter. Also I have bern experiencing stomach pains that prevent me from eating. I immediatly feel full and have extreme pain. Please help

--------------------------------------------------------------------------------
🔄 [1. AFTER CONTRACTION EXPANSION] (e.g., don't -> do not):
lump under left nipple and stomach pain (male) Hi,I have recently noticed a few weeks ago a lump under my nipple, it hurts to touch and is about the size of a quarter. Also I have bern experiencing stomach pains that prevent me from eating. I immediatly feel full and have extreme pain. Please help

--------------------------------------------------------------------------------
🔄 [2. AFTER PUNCTUATION SPACING ADJUSTMENT] (e.g., Hi,I -> Hi, I):
lump under left nipple and stomach pa

In [ ]:
# Load execution models
nlp = spacy.load("en_core_web_md")
nlp_medical = spacy.load("en_ner_bc5cdr_md")

print("================================================================================")
print("🧬 ADVANCED/BASIC HYBRID TECHNIQUE: CLINICAL NAMED ENTITY RECOGNITION")
print("================================================================================")
patient_text = df_clean_final['clean_input'].iloc[2]
doctor_text = df_clean_final['output'].iloc[2]

doc_patient_med = nlp_medical(patient_text)
doc_doctor_med = nlp_medical(doctor_text)

print(f"\n🔴 Cleaned Patient Input:\n\"{patient_text}\"")
print("\nDetected Clinical Entities:")
if not doc_patient_med.ents:
    print("   No clinical entities detected.")
else:
    for ent in doc_patient_med.ents:
        print(f"   👉 '{ent.text}' ➔ Clinical Type: {ent.label_}")

print(f"\n🟢 Raw Doctor Output:\n\"{doctor_text[:250]}...\"")
print("\nDetected Clinical Entities:")
if not doc_doctor_med.ents:
    print("   No clinical entities detected.")
else:
    for ent in doc_doctor_med.ents:
        print(f"   👉 '{ent.text}' ➔ Clinical Type: {ent.label_}")

print("\n" + "="*80 + "\n")

print("================================================================================")
print("🧠 BASIC TECHNIQUE: TEXT SIMILARITY (SENTENCE EMBEDDINGS & COSINE)")
print("================================================================================")
doc_patient = nlp(patient_text)
doc_doctor = nlp(doctor_text)
unrelated_doctor_text = df_clean_final['output'].iloc[0]
doc_unrelated = nlp(unrelated_doctor_text)

vec_patient = doc_patient.vector.reshape(1, -1)
vec_correct_doc = doc_doctor.vector.reshape(1, -1)
vec_unrelated_doc = doc_unrelated.vector.reshape(1, -1)

sim_correct = cosine_similarity(vec_patient, vec_correct_doc)[0][0]
sim_unrelated = cosine_similarity(vec_patient, vec_unrelated_doc)[0][0]

print(f"✅ Cosine Similarity (Patient vs. Correct Doctor Match)  : {sim_correct:.4f}")
print(f"❌ Cosine Similarity (Patient vs. Unrelated Doctor Match): {sim_unrelated:.4f}\n")

print("================================================================================")
print("🎭 BASIC TECHNIQUE: SENTIMENT ANALYSIS FOR PATIENT SATISFACTION")
print("================================================================================")
painful_patient_text = df_clean_final['clean_input'].iloc[3]
blob_analysis = TextBlob(painful_patient_text)
sentiment_score = blob_analysis.sentiment.polarity

print(f"📄 Analyzing Patient Query:\n\"{painful_patient_text}\"")
print(f"\n🎭 Calculated Sentiment Polarity Score: {sentiment_score:.4f}")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject